# 10 — Ablation matrix (TABLE 2) and provenance disaggregation (TABLE 3)

Two things the IDF still calls for, both now buildable:

**TABLE 2 — ablation matrix.** Disable each of the three built signals
(survival, cash-flow, anomaly) in turn and measure the fused AUC for every
combination, via the same 5-fold CV as notebook 09. This establishes
whether the fusion's effect comes from the combination or from one signal
carrying all the weight. NLP is a 4th planned dimension, not yet buildable
(no text corpus) -- this table is honestly scoped to 3 signals.

**TABLE 3 — provenance disaggregation.** Honest framing up front: the
survival model built in notebook 03 does NOT currently consume any
synthesized (Tier 2) features -- it only uses real funding/sector data.
So the IDF's original phrasing ("separate performance on real vs.
synthesised values") doesn't yet apply to the survival model as built.
What this notebook does instead is the fair, buildable version of that
question: take the same companies, fit the survival model twice -- once
with only the original real features, once with synthesized burn features
added -- and compare concordance on the identical subset. That isolates
whether Tier 2's synthesis actually helps Tier 3, which is the real claim
being tested.

Reads: `data/processed/outcomes.csv`, `data/processed/synthetic_trajectories.csv`,
`data/processed/survival_scored.csv`, `data/processed/cashflow_projections.csv`,
`data/processed/spend_anomalies.csv`
Writes: `reports/table2_ablation_matrix.csv`, `reports/table3_provenance_disaggregation.csv`

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

PROCESSED = "../data/processed"
REPORTS = "../reports"

survival = pd.read_csv(f"{PROCESSED}/survival_scored.csv")
cashflow = pd.read_csv(f"{PROCESSED}/cashflow_projections.csv")
anomalies = pd.read_csv(f"{PROCESSED}/spend_anomalies.csv")
traj = pd.read_csv(f"{PROCESSED}/synthetic_trajectories.csv")

anomaly_agg = anomalies.groupby("object_id")["is_anomaly"].mean().rename("anomaly_rate").reset_index()
merged = survival.merge(cashflow, left_on="id", right_on="object_id", how="inner") \
                  .merge(anomaly_agg, on="object_id", how="left")
merged["anomaly_rate"] = merged["anomaly_rate"].fillna(0)
merged["cashflow_urgency"] = 1 / (1 + merged["months_to_zero_projected"].clip(lower=0) / 12)
print(f"[load] {len(merged):,} companies with all signals for the ablation matrix")

[load] 1,892 companies with all signals for the ablation matrix


## TABLE 2 -- ablation matrix over the 3 built signals

In [2]:
ALL_SIGNALS = ["p_exhaust_6m", "cashflow_urgency", "anomaly_rate"]
y = merged["event"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7) if y.value_counts().min() >= 5 else 5

rows = []
for r in range(1, len(ALL_SIGNALS) + 1):
    for combo in combinations(ALL_SIGNALS, r):
        X_combo = merged[list(combo)].fillna(0)
        scores = cross_val_score(LogisticRegression(), X_combo, y, cv=cv, scoring="roc_auc")
        rows.append({
            "signals_enabled": " + ".join(combo),
            "n_signals": r,
            "auc_mean": scores.mean(),
            "auc_std": scores.std(),
        })

ablation = pd.DataFrame(rows).sort_values("auc_mean", ascending=False)
ablation.to_csv(f"{REPORTS}/table2_ablation_matrix.csv", index=False)
print("[TABLE 2] ablation matrix (all non-empty subsets of the 3 built signals):\n")
print(ablation.to_string(index=False))
print(f"\n[saved] {REPORTS}/table2_ablation_matrix.csv")
print("\nNOTE: a 4th row for 'distress language' cannot be added until Phase 2b's text")
print("corpus exists -- this table should be re-run with 15 rows (2^4 - 1) once it does.")

[TABLE 2] ablation matrix (all non-empty subsets of the 3 built signals):

                               signals_enabled  n_signals  auc_mean  auc_std
                                  p_exhaust_6m          1  0.753072 0.017861
                   p_exhaust_6m + anomaly_rate          2  0.741196 0.014444
                                  anomaly_rate          1  0.637514 0.014904
p_exhaust_6m + cashflow_urgency + anomaly_rate          3  0.622331 0.056736
               cashflow_urgency + anomaly_rate          2  0.593903 0.038269
               p_exhaust_6m + cashflow_urgency          2  0.561317 0.029665
                              cashflow_urgency          1  0.506800 0.018182

[saved] ../reports/table2_ablation_matrix.csv

NOTE: a 4th row for 'distress language' cannot be added until Phase 2b's text
corpus exists -- this table should be re-run with 15 rows (2^4 - 1) once it does.


## TABLE 3 -- does Tier 2 synthesis actually help Tier 3? (provenance disaggregation)

Fit the same Cox model architecture twice, on the identical subset of
companies, with and without synthesized-burn-derived features. This is
an apples-to-apples test of whether synthesis adds real signal.

In [3]:
synth_summary = traj.groupby("object_id")["synthesized_spend"].agg(
    synth_mean_spend="mean", synth_spend_volatility="std"
).reset_index()
synth_summary["synth_spend_volatility"] = synth_summary["synth_spend_volatility"].fillna(0)
print(f"[synthesis] summary features available for {len(synth_summary):,} companies")

[synthesis] summary features available for 11,050 companies


In [4]:
# Rebuild the same base feature set notebook 03 used, restricted to companies that
# also have synthesis available, so both models below are fit on IDENTICAL rows.
base_cols = [c for c in survival.columns if c not in
             ["p_exhaust_6m", "p_exhaust_12m"]]
base = survival[base_cols].copy()

enriched = base.merge(synth_summary, left_on="id", right_on="object_id", how="inner").drop(columns=["object_id"])
enriched["log_synth_spend"] = np.log1p(enriched["synth_mean_spend"])
enriched["log_synth_volatility"] = np.log1p(enriched["synth_spend_volatility"])

without_synthesis_cols = [c for c in base.columns if c not in ["id", "duration_months", "event"]]
with_synthesis_cols = without_synthesis_cols + ["log_synth_spend", "log_synth_volatility"]

def fit_and_score(df, feature_cols):
    cph = CoxPHFitter(penalizer=0.1)
    fit_df = df[feature_cols + ["duration_months", "event"]].dropna()
    cph.fit(fit_df, duration_col="duration_months", event_col="event")
    c_idx = concordance_index(fit_df["duration_months"], -cph.predict_partial_hazard(fit_df), fit_df["event"])
    return c_idx, len(fit_df)

c_without, n_without = fit_and_score(enriched, without_synthesis_cols)
c_with, n_with = fit_and_score(enriched, with_synthesis_cols)

result = pd.DataFrame([
    {"model": "Real/measured features only", "n_companies": n_without, "concordance_index": c_without},
    {"model": "Real features + synthesized burn features", "n_companies": n_with, "concordance_index": c_with},
])
result.to_csv(f"{REPORTS}/table3_provenance_disaggregation.csv", index=False)
print("[TABLE 3] same companies, with vs. without synthesized features:\n")
print(result.to_string(index=False))
print(f"\n[saved] {REPORTS}/table3_provenance_disaggregation.csv")
print()
if c_with > c_without:
    print("[TABLE 3] synthesized burn features improve concordance on this identical subset --")
    print("[TABLE 3] genuine evidence that Tier 2 synthesis contributes real signal to Tier 3.")
else:
    print("[TABLE 3] synthesized burn features do NOT improve concordance here -- report this plainly.")
    print("[TABLE 3] Possible reasons: subset too small, synthesis quality (see notebook 06's MAPE),")
    print("[TABLE 3] or the aggregation (mean/volatility) is too coarse a summary of the trajectory.")

[TABLE 3] same companies, with vs. without synthesized features:

                                    model  n_companies  concordance_index
              Real/measured features only        11021           0.810540
Real features + synthesized burn features        11021           0.822659

[saved] ../reports/table3_provenance_disaggregation.csv

[TABLE 3] synthesized burn features improve concordance on this identical subset --
[TABLE 3] genuine evidence that Tier 2 synthesis contributes real signal to Tier 3.
